# Window Functions

## What is a Window Function?
- Performs a calculation **across a set of rows** related to the current row
- Unlike `groupBy`, the original rows are **not collapsed** — you keep every row
- Think of it like: *"rank this row compared to others in the same group"*
### The building block: `Window.partitionBy().orderBy()`
- **`partitionBy`** — defines the group (like GROUP BY)
- **`orderBy`** — defines the sort order within each group

## Setup — Create Schema and Load Flights to Delta Tables
This notebook uses the `window_demo` schema to keep its tables isolated from other notebooks.
Narrowed to the Baton Rouge area airports (MSY/BTR) so this runs fast live.
Run these cells once before the rest of the demo.

In [ ]:
# Create an isolated schema for this demo
spark.sql("CREATE SCHEMA IF NOT EXISTS window_demo")
print("Schema ready: window_demo")

In [ ]:
from pyspark.sql.functions import col, year, month, to_date

# Load all three years of flights, narrowed to the Baton Rouge area, with year/month columns for the time-series sections below
(
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("Files/flights/flights_*.csv")
    .withColumn("FlightDate", to_date(col("FlightDate")))
    .withColumn("year", year(col("FlightDate")))
    .withColumn("month", month(col("FlightDate")))
    .filter(col("Origin").isin("MSY", "BTR"))
    .write
    .mode("overwrite")
    .saveAsTable("window_demo.flights")
)

print(f"flights table created: {spark.table('window_demo.flights').count()} rows")

In [ ]:
# Load airlines.csv and save as a managed Delta table in window_demo schema
(
    spark.read
    .format("csv")
    .option("header", "true")
    .load("Files/flights/airlines.csv")
    .write
    .mode("overwrite")
    .saveAsTable("window_demo.airlines")
)

print(f"airlines table created: {spark.table('window_demo.airlines').count()} rows")

### Load Data

In [ ]:
from pyspark.sql import Window
from pyspark.sql.functions import col, rank, dense_rank, row_number, lag, lead, sum, avg, count, ntile, first, last, datediff

# Load Baton Rouge area flights from the window_demo schema
flights = spark.table("window_demo.flights")

display(flights.limit(5))

## 1. Ranking Functions
| Function | Behavior with ties |
|----------|--------------------|
| `rank()` | Leaves gaps after ties (1, 1, 3) |
| `dense_rank()` | No gaps (1, 1, 2) |
| `row_number()` | Always unique — ties broken arbitrarily |

In [ ]:
# Rank flights within each airline by arrival delay (worst first)
window_by_airline = Window.partitionBy("Reporting_Airline").orderBy(col("ArrDelay").desc())

ranked = (
    flights
    .withColumn("rank",        rank().over(window_by_airline))
    .withColumn("dense_rank",  dense_rank().over(window_by_airline))
    .withColumn("row_number",  row_number().over(window_by_airline))
    .select("Reporting_Airline", "FlightDate", "Origin", "Dest", "ArrDelay", "rank", "dense_rank", "row_number")
)

display(ranked.filter(col("Reporting_Airline") == "WN").orderBy("rank"))

### Top 1 per group — the classic use case
Equivalent of SQL: `WHERE rank = 1`

In [ ]:
# Get the single most-delayed flight for each airline
worst_per_airline = (
    flights
    .withColumn("rank", rank().over(window_by_airline))
    .filter(col("rank") == 1)
    .select("Reporting_Airline", "FlightDate", "Origin", "Dest", "ArrDelay")
    .orderBy(col("ArrDelay").desc())
)

display(worst_per_airline.limit(10))

## 2. Lag and Lead
- **`lag()`** — looks at the **previous** row's value
- **`lead()`** — looks at the **next** row's value
- Great for period-over-period comparisons

In [ ]:
from pyspark.sql.functions import count as spark_count

# Count flights per month across the three years
flights_per_month = (
    flights
    .groupBy("year", "month")
    .agg(spark_count("FlightDate").alias("flight_count"))
    .orderBy("year", "month")
)

display(flights_per_month)

In [ ]:
from pyspark.sql.functions import round as spark_round

# Compare each month to the one before and after it
window_by_month = Window.orderBy("year", "month")

mom = (
    flights_per_month
    .withColumn("prev_month_count", lag("flight_count", 1).over(window_by_month))
    .withColumn("next_month_count", lead("flight_count", 1).over(window_by_month))
    .withColumn(
        "mom_change",
        spark_round((col("flight_count") - col("prev_month_count")) / col("prev_month_count") * 100, 1)
    )
)

display(mom)

## 3. Running / Cumulative Aggregates
- Aggregate functions (`sum`, `avg`, `count`) can run over a window
- **`rowsBetween`** controls the frame — which rows are included in the calculation
  - `Window.unboundedPreceding` = from the very first row
  - `Window.currentRow` = up to and including the current row

In [ ]:
# Running total of flights, ordered by month
running_window = (
    Window
    .orderBy("year", "month")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

running_total = (
    flights_per_month
    .withColumn("running_total", sum("flight_count").over(running_window))
)

display(running_total)

In [ ]:
# Rolling 3-month average
rolling_window = (
    Window
    .orderBy("year", "month")
    .rowsBetween(-2, Window.currentRow)  # current row + 2 rows before
)

rolling_avg = (
    flights_per_month
    .withColumn("rolling_3mo_avg", spark_round(avg("flight_count").over(rolling_window), 1))
)

display(rolling_avg)

## 4. ntile — Divide rows into equal buckets
- `ntile(4)` = divide into quartiles
- `ntile(10)` = deciles
- Useful for segmentation (top 25%, bottom 25%, etc.)

In [ ]:
# Assign each flight to a delay quartile, within its own airline
quartile_window = Window.partitionBy("Reporting_Airline").orderBy("ArrDelay")

quartiles = (
    flights
    .withColumn("delay_quartile", ntile(4).over(quartile_window))
    .select("Reporting_Airline", "FlightDate", "Origin", "Dest", "ArrDelay", "delay_quartile")
)

# Worst-quartile (most delayed) flights for Southwest
display(
    quartiles
    .filter((col("Reporting_Airline") == "WN") & (col("delay_quartile") == 4))
    .orderBy(col("ArrDelay").desc())
)

## 5. Named Windows — reuse the same spec

In [ ]:
# Define the window spec once and reuse it across multiple columns
airline_window = Window.partitionBy("Reporting_Airline").orderBy(col("Distance").desc())

enriched = (
    flights
    .withColumn("rank_in_airline",         rank().over(airline_window))
    .withColumn("avg_distance_airline",    spark_round(avg("Distance").over(Window.partitionBy("Reporting_Airline")), 1))
    .withColumn("distance_vs_airline_avg", spark_round(col("Distance") / avg("Distance").over(Window.partitionBy("Reporting_Airline")), 2))
    .select("Reporting_Airline", "FlightDate", "Origin", "Dest", "Distance", "rank_in_airline", "avg_distance_airline", "distance_vs_airline_avg")
    .filter(col("rank_in_airline") <= 3)
    .orderBy("Reporting_Airline", "rank_in_airline")
)

display(enriched.limit(15))

## 6. first() and last() — grab boundary values
Our dataset only spans three years, so every airline's first/last flight sits near the
edges of that window — the point here is the syntax, not the spread.

In [ ]:
# For each airline, show its earliest and most recent flight date in the dataset
airline_history_window = Window.partitionBy("Reporting_Airline").orderBy("FlightDate")
airline_history_window_desc = Window.partitionBy("Reporting_Airline").orderBy(col("FlightDate").desc())

first_last = (
    flights
    .withColumn("first_date",  first("FlightDate").over(airline_history_window))
    .withColumn("latest_date", first("FlightDate").over(airline_history_window_desc))
    .select("Reporting_Airline", "first_date", "latest_date")
    .dropDuplicates(["Reporting_Airline"])
    .withColumn("days_observed", datediff(col("latest_date"), col("first_date")))
    .orderBy(col("days_observed").desc())
)

display(first_last.limit(10))

---
## Wrap-up
That's the full set: ranking, lag/lead, running aggregates, ntile segmentation, named window reuse,
and first/last boundary values — all without collapsing a single row. Last notebook in the sequence.